# Detección de fraudes con programación genética
Taller de programación genética. Ejecuta las celdas en orden (*Entorno de ejecución → Ejecutar todo*). Solo usa numpy, pandas, scikit-learn y matplotlib (ya incluidos en Colab).

**Contenido:** 1) Requerimientos · 2) Datos v1 (aleatorio) y v2 (realista) · 3) Motor de programación genética · 4) Entrenamiento y resultados · 5) Pruebas · 6) Explicación en HTML

## 1. Requerimientos
- RF-01 Generar aleatoriamente un archivo con monto (pesos), hora del día, distancia a la residencia, ingreso mensual y fraude (sí/no).
- RF-02 Revisarlo y modificarlo para que sea más realista.
- RF-03 Encontrar un modelo que determine un posible fraude (programación genética).
- RF-04 Evaluar con datos no vistos. RNF-01 Reproducible (semillas fijas).

**Ticket FRAUDE-01 · criterios de aceptación:** CSV con las 5 columnas; v2 con fraude raro, nocturno, lejano y alto frente al ingreso; en v1 el modelo no supera el azar (AUC ≈ 0.5); en v2 AUC de prueba > 0.65 con fórmula legible; pruebas en verde.

## 2. Datos
`v1` es el archivo aleatorio ingenuo; `v2` es la versión corregida con criterio realista.

In [ ]:
import json, random, html
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

COLS = ["monto", "hora", "distancia_km", "ingreso_mensual", "fraude"]

def generar_v1(n=5000, seed=42):
    """Todo uniforme e independiente; la etiqueta es una moneda al aire."""
    r = np.random.default_rng(seed)
    return pd.DataFrame({
        "monto": r.integers(1_000, 5_000_000, n),
        "hora": r.integers(0, 24, n),
        "distancia_km": r.uniform(0, 500, n).round(1),
        "ingreso_mensual": r.integers(1_300_000, 20_000_000, n),
        "fraude": r.choice(["no", "si"], n)})

def generar_v2(n=5000, seed=42, tasa_objetivo=0.04):
    """Distribuciones realistas, fraude raro (~4%) y dependiente de las variables."""
    r = np.random.default_rng(seed)
    ingreso = np.clip(r.lognormal(np.log(2_800_000), 0.6, n), 1_300_000, 40_000_000)
    # El gasto típico es una fracción pequeña del ingreso (lognormal)
    monto = np.clip(ingreso * r.lognormal(np.log(0.03), 1.0, n), 2_000, None)
    # Horas: pico diurno y vespertino, pocas de madrugada
    pesos = np.array([1,1,1,1,1,2,4,6,8,8,8,9,10,9,8,8,8,9,10,9,7,5,3,2], float)
    hora = r.choice(24, n, p=pesos / pesos.sum())
    # Distancia a la residencia: casi siempre cerca, cola larga
    dist = np.where(r.random(n) < 0.85, r.exponential(6, n), r.exponential(80, n))
    ratio = monto / ingreso
    noche = ((hora <= 5) | (hora == 23)).astype(float)
    logit = (-7.0 + 1.6 * noche + 1.1 * np.log1p(dist / 10)
             + 1.4 * np.log1p(ratio * 5) + 0.3 * r.normal(size=n))
    # Calibra el intercepto para acercarse a la tasa objetivo
    lo, hi = -6.0, 6.0
    for _ in range(40):
        mid = (lo + hi) / 2
        p = 1 / (1 + np.exp(-(logit + mid)))
        lo, hi = (mid, hi) if p.mean() < tasa_objetivo else (lo, mid)
    p = 1 / (1 + np.exp(-(logit + mid)))
    fraude = np.where(r.random(n) < p, "si", "no")
    return pd.DataFrame({
        "monto": monto.round(-2).astype(int), "hora": hora,
        "distancia_km": dist.round(1),
        "ingreso_mensual": ingreso.round(-3).astype(int), "fraude": fraude})

v1 = generar_v1(); v2 = generar_v2()
v1.to_csv('datos_v1_aleatorio.csv', index=False)
v2.to_csv('datos_v2_realista.csv', index=False)
print('Tasa de fraude v1: %.3f | v2: %.3f' % ((v1.fraude=='si').mean(), (v2.fraude=='si').mean()))
display(v2.head()); display(v2.describe().round(1))

In [ ]:
# Revisión visual de v2: el fraude es raro y depende de hora, distancia y monto/ingreso
f = v2.fraude == 'si'
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
v2.groupby('hora').apply(lambda d: (d.fraude=='si').mean()).plot.bar(ax=ax[0], title='Tasa de fraude por hora')
ax[1].hist([v2.distancia_km[~f].clip(upper=150), v2.distancia_km[f].clip(upper=150)], bins=30, density=True, label=['normal','fraude']); ax[1].legend(); ax[1].set_title('Distancia (km)')
ax[2].hist([np.log10((v2.monto/v2.ingreso_mensual)[~f]), np.log10((v2.monto/v2.ingreso_mensual)[f])], bins=30, density=True, label=['normal','fraude']); ax[2].legend(); ax[2].set_title('log10(monto/ingreso)')
plt.tight_layout(); plt.show()

## 3. Motor de programación genética
Árboles de expresión (`+ − × ÷ max min`), aptitud = AUC − 0.002·nodos, torneo, elitismo, cruce y mutación de subárboles. Umbral que maximiza F1 en entrenamiento.

In [ ]:
FEATURES = ["monto_log", "ingreso_log", "ratio_log", "dist_log", "hora_sin", "hora_cos"]
OPS = {"+": np.add, "-": np.subtract, "*": np.multiply,
       "/": lambda a, b: np.where(np.abs(b) > 1e-3, a / np.where(np.abs(b) > 1e-3, b, 1), 1.0),
       "max": np.maximum, "min": np.minimum}

def preparar(df, stats=None):
    """Deriva y estandariza variables. `stats` evita fuga de datos (se calcula solo en train)."""
    X = pd.DataFrame({
        "monto_log": np.log1p(df.monto), "ingreso_log": np.log1p(df.ingreso_mensual),
        "ratio_log": np.log1p(df.monto / df.ingreso_mensual * 100),
        "dist_log": np.log1p(df.distancia_km),
        "hora_sin": np.sin(2 * np.pi * df.hora / 24), "hora_cos": np.cos(2 * np.pi * df.hora / 24)})
    if stats is None:
        stats = (X.mean(), X.std().replace(0, 1))
    return ((X - stats[0]) / stats[1]), stats

# Árbol: ("op", izq, der) | ("var", nombre) | ("const", valor)
def aleatorio(rng, prof, completo):
    if prof == 0 or (not completo and rng.random() < 0.3):
        if rng.random() < 0.8: return ("var", rng.choice(FEATURES))
        return ("const", round(rng.uniform(-2, 2), 2))
    op = rng.choice(list(OPS))
    return (op, aleatorio(rng, prof - 1, completo), aleatorio(rng, prof - 1, completo))

def evaluar(t, X):
    if t[0] == "var": return X[t[1]].to_numpy() if hasattr(X, "to_numpy") else X[t[1]]
    if t[0] == "const": return np.full(len(next(iter(X.values())) if isinstance(X, dict) else X), t[1])
    return OPS[t[0]](evaluar(t[1], X), evaluar(t[2], X))

def tamano(t): return 1 if t[0] in ("var", "const") else 1 + tamano(t[1]) + tamano(t[2])

def a_texto(t):
    if t[0] == "var": return t[1]
    if t[0] == "const": return str(t[1])
    if t[0] in ("max", "min"): return f"{t[0]}({a_texto(t[1])}, {a_texto(t[2])})"
    return f"({a_texto(t[1])} {t[0]} {a_texto(t[2])})"

def auc(y, s):
    """AUC por rangos (Mann-Whitney)."""
    y = np.asarray(y); s = np.nan_to_num(np.asarray(s, float), nan=0, posinf=1e9, neginf=-1e9)
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0: return 0.5
    rangos = pd.Series(s).rank().to_numpy()
    return float((rangos[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def nodos(t, ruta=()):
    yield ruta, t
    if t[0] not in ("var", "const"):
        yield from nodos(t[1], ruta + (1,)); yield from nodos(t[2], ruta + (2,))

def reemplazar(t, ruta, sub):
    if not ruta: return sub
    l = list(t); l[ruta[0]] = reemplazar(t[ruta[0]], ruta[1:], sub); return tuple(l)

def cruce(a, b, rng):
    ra, _ = rng.choice(list(nodos(a))); _, sb = rng.choice(list(nodos(b)))
    return reemplazar(a, ra, sb)

def mutar(t, rng):
    r, _ = rng.choice(list(nodos(t)))
    return reemplazar(t, r, aleatorio(rng, 2, False))

def mejor_umbral(y, s):
    """Umbral que maximiza F1 sobre `y`."""
    cand = np.unique(np.quantile(s, np.linspace(0.5, 0.995, 120)))
    return max(cand, key=lambda u: metricas(y, s > u)["f1"])

def metricas(y, pred):
    y = np.asarray(y); pred = np.asarray(pred).astype(int)
    tp = int(((y == 1) & (pred == 1)).sum()); fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum()); tn = int(((y == 0) & (pred == 0)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn, "precision": p, "recall": r,
            "f1": 2 * p * r / (p + r) if p + r else 0.0}

class GPFraude:
    def __init__(self, poblacion=300, generaciones=40, prof_max=6, penal=0.002, seed=0):
        self.N, self.G, self.prof, self.penal = poblacion, generaciones, prof_max, penal
        self.rng = random.Random(seed); self.historial = []

    def _fit(self, t, X, y):
        s = evaluar(t, X)
        return auc(y, s) - self.penal * tamano(t) if np.all(np.isfinite(s)) else -1

    def entrenar(self, X, y):
        y = np.asarray(y)
        pob = [aleatorio(self.rng, 1 + i % 4, i % 2 == 0) for i in range(self.N)]
        for g in range(self.G):
            fs = [self._fit(t, X, y) for t in pob]
            orden = sorted(range(self.N), key=lambda i: -fs[i])
            self.historial.append({"gen": g, "mejor": fs[orden[0]], "media": float(np.mean(fs))})
            nueva = [pob[i] for i in orden[:5]]  # elitismo
            def torneo():
                c = self.rng.sample(range(self.N), 5); return pob[max(c, key=lambda i: fs[i])]
            while len(nueva) < self.N:
                u = self.rng.random()
                h = cruce(torneo(), torneo(), self.rng) if u < 0.8 else mutar(torneo(), self.rng)
                if tamano(h) <= 2 ** self.prof: nueva.append(h)
            pob = nueva
        fs = [self._fit(t, X, y) for t in pob]
        self.arbol = pob[int(np.argmax(fs))]
        self.umbral = mejor_umbral(y, evaluar(self.arbol, X))
        return self

    def puntaje(self, X): return evaluar(self.arbol, X)
    def predecir(self, X): return (self.puntaje(X) > self.umbral).astype(int)

## 4. Entrenamiento y resultados
División 70/30 estratificada; comparación contra regresión logística.

In [ ]:
def correr(df, nombre, seed=1):
    y = (df.fraude == "si").astype(int).to_numpy()
    tr, te = train_test_split(df.index, test_size=0.3, stratify=y, random_state=seed)
    Xtr, st = preparar(df.loc[tr]); Xte, _ = preparar(df.loc[te], st)
    ytr, yte = y[tr], y[te]
    gp = GPFraude(seed=seed).entrenar(Xtr, ytr)
    s = gp.puntaje(Xte); m = metricas(yte, gp.predecir(Xte))
    lr = LogisticRegression(class_weight="balanced", max_iter=1000).fit(Xtr, ytr)
    sl = lr.decision_function(Xte)
    umb = np.quantile(lr.decision_function(Xtr), 1 - ytr.mean() * 1.5)
    ml = metricas(yte, sl > umb)
    return {"nombre": nombre, "tasa_fraude": float(y.mean()),
            "gp": {"auc": auc(yte, s), **m, "formula": a_texto(gp.arbol), "nodos": tamano(gp.arbol),
                   "umbral": float(gp.umbral)},
            "logistica": {"auc": auc(yte, sl), **ml}, "historial": gp.historial}

res = [correr(v1, 'v1 aleatorio'), correr(v2, 'v2 realista')]
for r in res:
    print(r['nombre'], '| fraude=%.3f' % r['tasa_fraude'])
    print('  GP  AUC=%.3f P=%.2f R=%.2f F1=%.2f' % (r['gp']['auc'], r['gp']['precision'], r['gp']['recall'], r['gp']['f1']))
    print('  LR  AUC=%.3f P=%.2f R=%.2f F1=%.2f' % (r['logistica']['auc'], r['logistica']['precision'], r['logistica']['recall'], r['logistica']['f1']))
    print('  Fórmula GP:', r['gp']['formula'], '> %.2f' % r['gp']['umbral'])

h = pd.DataFrame(res[1]['historial'])
plt.plot(h.gen, h.mejor, label='mejor'); plt.plot(h.gen, h.media, '--', label='media')
plt.xlabel('generación'); plt.ylabel('aptitud'); plt.title('Convergencia (v2)'); plt.legend(); plt.show()

## 5. Pruebas
11 pruebas unitarias (datos, métricas, operadores genéticos, reproducibilidad).

In [ ]:
import unittest
class TestDatos(unittest.TestCase):
    def test_columnas_y_reproducible(self):
        a, b = generar_v2(500, seed=1), generar_v2(500, seed=1)
        self.assertEqual(list(a.columns), COLS); self.assertTrue(a.equals(b))
    def test_rangos_realistas(self):
        d = generar_v2(3000)
        self.assertTrue(d.hora.between(0, 23).all()); self.assertTrue((d.monto > 0).all())
        self.assertTrue((d.distancia_km >= 0).all()); self.assertTrue((d.ingreso_mensual >= 1_300_000).all())
    def test_fraude_raro_y_nocturno(self):
        d = generar_v2(20000); f = d.fraude == "si"
        self.assertTrue(0.02 < f.mean() < 0.07)
        noche = (d.hora <= 5) | (d.hora == 23)
        self.assertGreater(f[noche].mean(), f[~noche].mean())
    def test_v1_sin_senal(self):
        d = generar_v1(4000); f = d.fraude == "si"
        self.assertAlmostEqual(f.mean(), 0.5, delta=0.05)

class TestGP(unittest.TestCase):
    def test_auc_basico(self):
        self.assertEqual(auc([0, 0, 1, 1], [1, 2, 3, 4]), 1.0)
        self.assertEqual(auc([0, 0, 1, 1], [4, 3, 2, 1]), 0.0)
        self.assertEqual(auc([0, 0, 0], [1, 2, 3]), 0.5)
    def test_metricas(self):
        m = metricas([1, 1, 0, 0], [1, 0, 1, 0])
        self.assertEqual((m["tp"], m["fp"], m["fn"], m["tn"]), (1, 1, 1, 1)); self.assertEqual(m["f1"], 0.5)
    def test_division_protegida(self):
        X = {"a": np.array([1.0, 2.0]), "b": np.array([0.0, 2.0])}
        self.assertTrue(np.all(np.isfinite(evaluar(("/", ("var", "a"), ("var", "b")), X))))
    def test_operadores_geneticos_validos(self):
        r = random.Random(0); a, b = aleatorio(r, 4, True), aleatorio(r, 4, True)
        for _ in range(50):
            self.assertGreaterEqual(tamano(cruce(a, b, r)), 1); self.assertGreaterEqual(tamano(mutar(a, r)), 1)
    def test_preparar_sin_fuga(self):
        d = generar_v2(600); X, st = preparar(d.iloc[:400]); Xt, _ = preparar(d.iloc[400:], st)
        self.assertAlmostEqual(float(X.mean().abs().max()), 0, places=6)
        self.assertEqual(list(Xt.columns), FEATURES)
    def test_gp_aprende_senal_y_ignora_ruido(self):
        d = generar_v2(3000, seed=3); y = (d.fraude == "si").astype(int).to_numpy()
        X, st = preparar(d.iloc[:2000]); Xt, _ = preparar(d.iloc[2000:], st)
        gp = GPFraude(poblacion=120, generaciones=15, seed=0).entrenar(X, y[:2000])
        self.assertGreater(auc(y[2000:], gp.puntaje(Xt)), 0.65)
        self.assertEqual(set(gp.predecir(Xt)) <= {0, 1}, True)
        self.assertTrue(isinstance(a_texto(gp.arbol), str))
    def test_gp_reproducible(self):
        d = generar_v2(800); y = (d.fraude == "si").astype(int).to_numpy(); X, _ = preparar(d)
        f = lambda: a_texto(GPFraude(poblacion=40, generaciones=5, seed=7).entrenar(X, y).arbol)
        self.assertEqual(f(), f())

suite = unittest.TestSuite([unittest.defaultTestLoader.loadTestsFromTestCase(k) for k in (TestDatos, TestGP)])
unittest.TextTestRunner(verbosity=1).run(suite)

## 6. Explicación en HTML
Genera `explicacion_fraude.html` con los resultados de esta ejecución y lo muestra aquí.

In [ ]:
v1r, v2r = res; g = v2r['gp']
hh = v2r['historial']; W, H = 560, 200
pts = lambda k: " ".join(f"{20+i*(W-40)/(len(hh)-1):.1f},{H-20-(x[k]-0.5)/0.3*(H-40):.1f}" for i, x in enumerate(hh))
fila = lambda r, k, n: f"<tr><td>{n}</td><td>{r['nombre']}</td><td>{r[k]['auc']:.3f}</td><td>{r[k]['precision']:.2f}</td><td>{r[k]['recall']:.2f}</td><td>{r[k]['f1']:.2f}</td></tr>"
rows = "".join(fila(r, k, n) for r in res for k, n in (("gp","Prog. genética"),("logistica","Reg. logística")))
page = f"""<!doctype html><html lang="es"><head><meta charset="utf-8"><title>Detección de fraudes con programación genética</title>
<style>body{{font:16px/1.6 system-ui,sans-serif;max-width:820px;margin:auto;padding:24px;color:#1c1c1e;background:#fff}}h2{{border-bottom:1px solid #ddd}}
pre{{background:#f5f6f8;padding:12px;border-radius:6px;overflow-x:auto}}code{{background:#f5f6f8;padding:1px 5px;border-radius:4px}}
table{{border-collapse:collapse;width:100%}}td,th{{border:1px solid #ddd;padding:6px 10px;text-align:left}}.k{{background:#f5f6f8;border-left:4px solid #1d6fd6;padding:10px 14px}}svg{{max-width:100%}}</style></head><body>
<h1>Detección de fraudes con programación genética</h1>
<p class="k"><b>Resumen.</b> Se generó un dataset aleatorio (v1) y se rehízo con reglas realistas (v2). La programación genética evolucionó una fórmula que puntúa el riesgo: AUC de prueba <b>{g['auc']:.2f}</b> en v2 y <b>{v1r['gp']['auc']:.2f}</b> (azar) en v1, lo que confirma que aprende señal real y no memoriza.</p>
<h2>1. Los datos</h2><p>Columnas: <code>monto</code> (COP), <code>hora</code>, <code>distancia_km</code>, <code>ingreso_mensual</code>, <code>fraude</code>.</p>
<p><b>v1:</b> todo uniforme e independiente, fraude 50/50 (irreal). <b>Cambios en v2:</b></p><ul>
<li>Ingreso lognormal (mediana ≈ 2,8 M COP, mínimo 1,3 M).</li><li>Monto como fracción lognormal del ingreso.</li><li>Hora con picos diurnos; pocas transacciones de madrugada.</li>
<li>Distancia: 85 % cerca (media 6 km) y 15 % lejos (media 80 km).</li><li>Fraude ≈ {v2r['tasa_fraude']*100:.1f} %, más probable de noche, lejos de casa o con monto/ingreso alto, con ruido.</li></ul>
<h2>2. El modelo</h2><p>Árboles de expresión con <code>+ − × ÷ max min</code> (división protegida) sobre variables estandarizadas: <code>monto_log, ingreso_log, ratio_log, dist_log, hora_sin, hora_cos</code> (la hora es cíclica).</p>
<ul><li><b>Aptitud:</b> AUC − 0,002 × nodos.</li><li><b>Población:</b> 300, 40 generaciones, torneo de 5, elitismo 5, cruce 80 % / mutación 20 %.</li>
<li><b>Decisión:</b> fraude si la fórmula supera el umbral que maximiza F1 en train.</li><li><b>Evaluación:</b> 70/30 estratificado, estandarización solo con train.</li></ul>
<svg viewBox="0 0 {W} {H}"><rect width="{W}" height="{H}" fill="none" stroke="#ddd"/><polyline fill="none" stroke="#1d6fd6" stroke-width="2" points="{pts('mejor')}"/>
<polyline fill="none" stroke="#888" stroke-dasharray="4" points="{pts('media')}"/><text x="24" y="16" font-size="12">Aptitud por generación: mejor (azul), media (gris)</text></svg>
<h2>3. Resultado</h2><pre>{html.escape(g['formula'])}  &gt;  {g['umbral']:.2f}  →  posible fraude</pre>
<table><tr><th>Modelo</th><th>Datos</th><th>AUC</th><th>Precisión</th><th>Recall</th><th>F1</th></tr>{rows}</table>
<p>Matriz de confusión GP en v2 (prueba): VP {g['tp']}, FP {g['fp']}, FN {g['fn']}, VN {g['tn']}.</p>
<h2>4. Lectura crítica</h2><ul><li>En v1 el AUC ≈ 0,5: no hay nada que aprender y el modelo lo reconoce.</li>
<li>En v2 la GP iguala a la regresión logística en AUC y entrega una fórmula legible.</li>
<li>El desempeño es moderado por el ruido deliberado: el fraude no es perfectamente predecible.</li>
<li>Con pocos fraudes en prueba las métricas varían entre semillas; conviene repetir con varias.</li>
<li>Los datos son sintéticos: el modelo redescubre reglas definidas por nosotros, no fraude real.</li></ul></body></html>"""
open('explicacion_fraude.html', 'w', encoding='utf-8').write(page)
from IPython.display import HTML, display
display(HTML(page))
# En Colab, para descargar los archivos generados:
# from google.colab import files; [files.download(f) for f in ['datos_v1_aleatorio.csv','datos_v2_realista.csv','explicacion_fraude.html']]